# CalendarBot con el modelo fine-tuneado (LoRA)

Mensaje en lenguaje natural → **Qwen3-4B-Instruct-2507 + LoRA** (`qwen3-4b-semantic-lora`, `checkpoint-300`) extrae los 6 campos semánticos → **`resolvedor_temporal`** calcula la fecha (el modelo nunca calcula fechas) → reglas para zona horaria y recurrencia → JSON de 9 campos → evento en **Google Calendar**.

**Antes de empezar**
1. Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU T4**.
2. El adapter: `adapter_config.json` y `adapter_model.safetensors` de `checkpoint-300`. O bien en tu Drive (acceso directo a la carpeta compartida `qwen3-4b-semantic-lora` en *Mi unidad*), o bien los subes en la celda 5.
3. `credentials.json` de Google (y opcional `token_google.json`) para la celda 4.

Ejecuta las celdas en orden. La última es el chat: escribe tu solicitud en `SOLICITUD` y ejecútala; cada ejecución es un mensaje.

In [ ]:
# @title 1. Instalar dependencias (mismas versiones con las que se entrenó el adapter)
!nvidia-smi -L
%pip install -q "transformers==5.17.0" "peft==0.21.0" "accelerate==1.15.0"
%pip uninstall -y -q torchao
import torch
assert torch.cuda.is_available(), "Sin GPU: cambia el entorno de ejecución a T4 y vuelve a correr esta celda"
print("GPU OK:", torch.cuda.get_device_name(0))

GPU 0: Tesla T4 (UUID: GPU-dfd80169-8bc3-47c3-ea98-1ebe77a6e586)
GPU OK: Tesla T4


In [ ]:
%%writefile calendario.py
"""
calendario.py - Lleva el JSON del modelo (contrato de 9 campos) a Outlook o Google Calendar.

Uso:
  python calendario.py <id_frase>    -> muestra los payloads de Graph y Google y escribe
                                        evento_<id>.ics a partir del ground truth de frases.json

Funciones:
  a_evento_graph(evento)                     -> payload para POST /me/calendars/{id}/events (Outlook)
  a_evento_google(evento)                    -> payload para POST /calendars/{id}/events (Google)
  a_ics(evento)                              -> mismo evento como texto iCalendar (respaldo)
  crear_en_outlook(evento, client_id, ...)   -> login (device code) + creacion en Outlook
  crear_en_google(evento, credenciales, ...) -> login (link + pegar URL) + creacion en Google

Decisiones de diseño:
  - Sin duracion_minutos se usa DURACION_DEFECTO (el calendario exige hora de termino).
  - Los participantes van en la descripcion del evento, no como invitados: la API exige
    correos y enviaria invitaciones reales.
  - Los eventos se crean en un calendario aparte llamado CALENDARIO.
"""

import json
import os
import sys
import uuid
import urllib.error
import urllib.parse
import urllib.request
from datetime import datetime, timedelta, timezone

SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

DURACION_DEFECTO = 30
CALENDARIO = "CalendarBot"
GRAPH_URL = "https://graph.microsoft.com/v1.0"
SCOPES = ["Calendars.ReadWrite"]

GOOGLE_API = "https://www.googleapis.com/calendar/v3"
GOOGLE_REDIRECT = "http://localhost"
# solo calendarios creados por la app + ver la lista para no duplicar CALENDARIO
GOOGLE_SCOPES = [
    "https://www.googleapis.com/auth/calendar.app.created",
    "https://www.googleapis.com/auth/calendar.calendarlist.readonly",
]

# IANA -> nombre Windows que usa Outlook
ZONAS_WINDOWS = {
    "America/Santiago": "Pacific SA Standard Time",
    "Europe/Madrid": "Romance Standard Time",
    "America/Argentina/Buenos_Aires": "Argentina Standard Time",
    "America/Lima": "SA Pacific Standard Time",
    "America/Bogota": "SA Pacific Standard Time",
    "America/Mexico_City": "Central Standard Time (Mexico)",
    "America/New_York": "Eastern Standard Time",
    "America/Los_Angeles": "Pacific Standard Time",
    "Europe/London": "GMT Standard Time",
    "UTC": "UTC",
}

DIAS_ICS = {
    "monday": "MO", "tuesday": "TU", "wednesday": "WE", "thursday": "TH",
    "friday": "FR", "saturday": "SA", "sunday": "SU",
}
DIAS_SEMANA = list(DIAS_ICS)  # indice = datetime.weekday()

# Chile: horario de verano desde el primer domingo de septiembre hasta el primer domingo de abril
VTIMEZONE_SANTIAGO = """BEGIN:VTIMEZONE
TZID:America/Santiago
BEGIN:STANDARD
DTSTART:19700405T000000
TZOFFSETFROM:-0300
TZOFFSETTO:-0400
RRULE:FREQ=YEARLY;BYMONTH=4;BYDAY=1SU
END:STANDARD
BEGIN:DAYLIGHT
DTSTART:19700906T000000
TZOFFSETFROM:-0400
TZOFFSETTO:-0300
RRULE:FREQ=YEARLY;BYMONTH=9;BYDAY=1SU
END:DAYLIGHT
END:VTIMEZONE"""


def _inicio_fin(evento):
    if not isinstance(evento, dict) or "error" in evento:
        raise ValueError("el modelo no entrego un evento valido")
    if not evento.get("fecha") or not evento.get("hora_inicio"):
        raise ValueError("falta fecha/hora: no se puede crear el evento")
    inicio = datetime.strptime(f"{evento['fecha']} {evento['hora_inicio']}", "%Y-%m-%d %H:%M")
    duracion = evento.get("duracion_minutos") or DURACION_DEFECTO
    return inicio, inicio + timedelta(minutes=int(duracion))


def _recurrencia(evento):
    return evento.get("recurrencia", evento.get("recurrence"))


def _dias_recurrencia(rec, inicio):
    return [d.lower() for d in (rec.get("dias") or [])] or [DIAS_SEMANA[inicio.weekday()]]


def _cuerpo(evento):
    participantes = evento.get("participantes") or []
    return f"Participantes: {', '.join(participantes)}" if participantes else ""


def a_evento_graph(evento):
    """JSON del contrato -> payload de Microsoft Graph (recurso event)."""
    inicio, fin = _inicio_fin(evento)
    zona = ZONAS_WINDOWS.get(evento.get("zona_horaria") or "America/Santiago", "UTC")
    fmt = "%Y-%m-%dT%H:%M:00"

    payload = {
        "subject": evento.get("titulo") or "Reunión",
        "start": {"dateTime": inicio.strftime(fmt), "timeZone": zona},
        "end": {"dateTime": fin.strftime(fmt), "timeZone": zona},
        "isReminderOn": evento.get("recordatorio_minutos") is not None,
    }
    if evento.get("recordatorio_minutos") is not None:
        payload["reminderMinutesBeforeStart"] = int(evento["recordatorio_minutos"])
    if evento.get("ubicacion"):
        payload["location"] = {"displayName": evento["ubicacion"]}
    if _cuerpo(evento):
        payload["body"] = {"contentType": "text", "content": _cuerpo(evento)}

    rec = _recurrencia(evento)
    if rec:
        frecuencia = rec.get("frecuencia", "weekly")
        patron = {"interval": int(rec.get("intervalo") or 1)}
        if frecuencia == "daily":
            patron["type"] = "daily"
        elif frecuencia == "monthly":
            patron.update(type="absoluteMonthly", dayOfMonth=inicio.day)
        else:
            patron.update(type="weekly", daysOfWeek=_dias_recurrencia(rec, inicio))
        rango = {"startDate": evento["fecha"]}
        if rec.get("hasta"):
            rango.update(type="endDate", endDate=rec["hasta"])
        elif rec.get("veces"):
            rango.update(type="numbered", numberOfOccurrences=int(rec["veces"]))
        else:
            rango["type"] = "noEnd"
        payload["recurrence"] = {"pattern": patron, "range": rango}

    return payload


def _escapar_ics(texto):
    return (str(texto).replace("\\", "\\\\").replace(";", "\\;")
            .replace(",", "\\,").replace("\n", "\\n"))


def _rrule(rec, inicio, tzid):
    """Recurrencia del contrato -> regla RRULE (RFC 5545), la usan .ics y Google."""
    frecuencia = rec.get("frecuencia", "weekly")
    regla = f"FREQ={frecuencia.upper()};INTERVAL={int(rec.get('intervalo') or 1)}"
    if frecuencia == "weekly":
        regla += ";BYDAY=" + ",".join(DIAS_ICS[d] for d in _dias_recurrencia(rec, inicio))
    if rec.get("hasta"):
        # con zona horaria el estandar exige UNTIL en UTC: fin del dia local (Chile = UTC-4 como maximo)
        desfase = 4 if tzid == "America/Santiago" else 0
        hasta = datetime.strptime(rec["hasta"], "%Y-%m-%d") + timedelta(hours=23 + desfase, minutes=59, seconds=59)
        regla += f";UNTIL={hasta.strftime('%Y%m%dT%H%M%S')}Z"
    elif rec.get("veces"):
        regla += f";COUNT={int(rec['veces'])}"
    return regla


def a_evento_google(evento):
    """JSON del contrato -> payload de Google Calendar (recurso Event). Google acepta zonas IANA."""
    inicio, fin = _inicio_fin(evento)
    zona = evento.get("zona_horaria") or "America/Santiago"
    fmt = "%Y-%m-%dT%H:%M:00"

    recordatorio = evento.get("recordatorio_minutos")
    payload = {
        "summary": evento.get("titulo") or "Reunión",
        "start": {"dateTime": inicio.strftime(fmt), "timeZone": zona},
        "end": {"dateTime": fin.strftime(fmt), "timeZone": zona},
        "reminders": {
            "useDefault": False,
            "overrides": [{"method": "popup", "minutes": int(recordatorio)}] if recordatorio is not None else [],
        },
    }
    if evento.get("ubicacion"):
        payload["location"] = evento["ubicacion"]
    if _cuerpo(evento):
        payload["description"] = _cuerpo(evento)
    rec = _recurrencia(evento)
    if rec:
        payload["recurrence"] = [f"RRULE:{_rrule(rec, inicio, zona)}"]
    return payload


def a_ics(evento):
    """JSON del contrato -> texto iCalendar (.ics) que Outlook importa con doble clic."""
    inicio, fin = _inicio_fin(evento)
    tzid = evento.get("zona_horaria") or "America/Santiago"
    fmt = "%Y%m%dT%H%M%S"

    lineas = ["BEGIN:VCALENDAR", "VERSION:2.0", "PRODID:-//CalendarBot//UdeC GAI//ES", "CALSCALE:GREGORIAN"]
    if tzid == "America/Santiago":
        lineas.append(VTIMEZONE_SANTIAGO)
    lineas += [
        "BEGIN:VEVENT",
        f"UID:{uuid.uuid4()}@calendarbot",
        f"DTSTAMP:{datetime.now(timezone.utc).strftime(fmt)}Z",
        f"DTSTART;TZID={tzid}:{inicio.strftime(fmt)}",
        f"DTEND;TZID={tzid}:{fin.strftime(fmt)}",
        f"SUMMARY:{_escapar_ics(evento.get('titulo') or 'Reunión')}",
    ]
    if evento.get("ubicacion"):
        lineas.append(f"LOCATION:{_escapar_ics(evento['ubicacion'])}")
    if _cuerpo(evento):
        lineas.append(f"DESCRIPTION:{_escapar_ics(_cuerpo(evento))}")

    rec = _recurrencia(evento)
    if rec:
        lineas.append(f"RRULE:{_rrule(rec, inicio, tzid)}")

    if evento.get("recordatorio_minutos") is not None:
        lineas += ["BEGIN:VALARM", "ACTION:DISPLAY", "DESCRIPTION:Recordatorio",
                   f"TRIGGER:-PT{int(evento['recordatorio_minutos'])}M", "END:VALARM"]
    lineas += ["END:VEVENT", "END:VCALENDAR"]
    return "\r\n".join("\r\n".join(l.split("\n")) for l in lineas) + "\r\n"


_TOKENS = {}


def _token(client_id, tenant):
    """Login con device code flow: imprime un codigo para pegar en microsoft.com/devicelogin."""
    if (client_id, tenant) in _TOKENS:
        return _TOKENS[(client_id, tenant)]
    import msal
    app = msal.PublicClientApplication(client_id, authority=f"https://login.microsoftonline.com/{tenant}")
    flujo = app.initiate_device_flow(scopes=SCOPES)
    if "user_code" not in flujo:
        raise RuntimeError(flujo.get("error_description", flujo))
    print(flujo["message"])
    res = app.acquire_token_by_device_flow(flujo)
    if "access_token" not in res:
        raise RuntimeError(res.get("error_description", res))
    _TOKENS[(client_id, tenant)] = res["access_token"]
    return res["access_token"]


def _pedir(metodo, url, token=None, cuerpo=None, formulario=None):
    """HTTP con stdlib: cuerpo JSON o formulario; devuelve la respuesta JSON."""
    headers = {}
    datos = None
    if token:
        headers["Authorization"] = f"Bearer {token}"
    if cuerpo is not None:
        datos = json.dumps(cuerpo).encode("utf-8")
        headers["Content-Type"] = "application/json"
    elif formulario is not None:
        datos = urllib.parse.urlencode(formulario).encode("utf-8")
        headers["Content-Type"] = "application/x-www-form-urlencoded"
    req = urllib.request.Request(url, data=datos, method=metodo, headers=headers)
    try:
        with urllib.request.urlopen(req) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"HTTP {e.code} en {url}: {e.read().decode('utf-8', 'replace')}") from e


def _graph(metodo, ruta, token, cuerpo=None):
    return _pedir(metodo, f"{GRAPH_URL}{ruta}", token, cuerpo)


def _respaldo_ics(evento, error, payload, destino):
    ruta = f"evento_{uuid.uuid4().hex[:8]}.ics"
    with open(ruta, "w", encoding="utf-8", newline="") as f:
        f.write(a_ics(evento))
    print(f"[AVISO] No se pudo crear en {destino} ({error}). Respaldo: {ruta}")
    return {"ics": ruta, "error": str(error), "payload": payload}


def _id_calendario(token):
    for cal in _graph("GET", "/me/calendars?$top=100", token)["value"]:
        if cal["name"] == CALENDARIO:
            return cal["id"]
    return _graph("POST", "/me/calendars", token, {"name": CALENDARIO})["id"]


def crear_en_outlook(evento, client_id=None, tenant=None, enviar=False):
    """
    Crea el evento en el calendario CALENDARIO de Outlook.
    enviar=False solo muestra el payload. Si falla el login o Graph, escribe un .ics de respaldo.
    """
    payload = a_evento_graph(evento)  # sin fecha/hora falla aqui: no hay respaldo posible
    if not enviar:
        print(json.dumps(payload, indent=2, ensure_ascii=False))
        return {"payload": payload}

    client_id = client_id or os.environ.get("CALENDARBOT_CLIENT_ID")
    tenant = tenant or os.environ.get("CALENDARBOT_TENANT", "consumers")
    try:
        if not client_id:
            raise RuntimeError("falta client_id (variable CALENDARBOT_CLIENT_ID)")
        token = _token(client_id, tenant)
        creado = _graph("POST", f"/me/calendars/{_id_calendario(token)}/events", token, payload)
        print(f"Evento creado en Outlook: {creado['webLink']}")
        return {"id": creado["id"], "webLink": creado["webLink"], "payload": payload}
    except Exception as e:
        return _respaldo_ics(evento, e, payload, "Outlook")


def _token_google(credenciales, ruta_token):
    """
    Login OAuth de Google para app de escritorio. El device flow de Google no admite scopes de
    Calendar, asi que se abre un link, se autoriza y se pega la URL de localhost a la que
    redirige el navegador. El refresh_token queda en ruta_token para no repetir el login
    (en modo prueba de Google vence a los 7 dias; entonces se pide login de nuevo).
    """
    with open(credenciales, "r", encoding="utf-8") as f:
        cliente = json.load(f)["installed"]  # credentials.json de un cliente "Desktop app"
    base = {"client_id": cliente["client_id"], "client_secret": cliente["client_secret"]}

    if os.path.exists(ruta_token):
        with open(ruta_token, "r", encoding="utf-8") as f:
            refresh = json.load(f).get("refresh_token")
        if refresh:
            try:
                res = _pedir("POST", cliente["token_uri"], formulario={
                    **base, "refresh_token": refresh, "grant_type": "refresh_token"})
                return res["access_token"]
            except RuntimeError as e:
                print(f"[AVISO] Token guardado no sirve ({e}); se pide login de nuevo.")

    url = cliente["auth_uri"] + "?" + urllib.parse.urlencode({
        "client_id": cliente["client_id"], "redirect_uri": GOOGLE_REDIRECT, "response_type": "code",
        "scope": " ".join(GOOGLE_SCOPES), "access_type": "offline", "prompt": "consent"})
    print("1. Abre este link e inicia sesion con tu cuenta Google:\n" + url)
    print("2. Al final el navegador muestra una pagina que no carga (localhost).")
    print("   Copia la direccion completa de la barra y pegala aqui.")
    pegado = input("URL: ").strip()
    codigo = urllib.parse.parse_qs(urllib.parse.urlparse(pegado).query).get("code", [pegado])[0]
    res = _pedir("POST", cliente["token_uri"], formulario={
        **base, "code": codigo, "redirect_uri": GOOGLE_REDIRECT, "grant_type": "authorization_code"})
    if res.get("refresh_token"):
        with open(ruta_token, "w", encoding="utf-8") as f:
            json.dump({"refresh_token": res["refresh_token"]}, f)
    return res["access_token"]


def _id_calendario_google(token):
    for cal in _pedir("GET", f"{GOOGLE_API}/users/me/calendarList", token).get("items", []):
        if cal.get("summary") == CALENDARIO:
            return cal["id"]
    return _pedir("POST", f"{GOOGLE_API}/calendars", token,
                  {"summary": CALENDARIO, "timeZone": "America/Santiago"})["id"]


def crear_en_google(evento, credenciales="credentials.json", ruta_token="token_google.json", enviar=False):
    """
    Crea el evento en el calendario CALENDARIO de Google Calendar.
    enviar=False solo muestra el payload. Si falla el login o la API, escribe un .ics de respaldo.
    """
    payload = a_evento_google(evento)  # sin fecha/hora falla aqui: no hay respaldo posible
    if not enviar:
        print(json.dumps(payload, indent=2, ensure_ascii=False))
        return {"payload": payload}

    try:
        token = _token_google(credenciales, ruta_token)
        id_cal = urllib.parse.quote(_id_calendario_google(token), safe="")
        creado = _pedir("POST", f"{GOOGLE_API}/calendars/{id_cal}/events", token, payload)
        print(f"Evento creado en Google Calendar: {creado['htmlLink']}")
        return {"id": creado["id"], "htmlLink": creado["htmlLink"], "payload": payload}
    except Exception as e:
        return _respaldo_ics(evento, e, payload, "Google Calendar")


if __name__ == "__main__":
    if len(sys.argv) != 2:
        print(__doc__)
        sys.exit(1)
    with open(os.path.join(SCRIPT_DIR, "frases.json"), "r", encoding="utf-8") as f:
        frases = {fr["id"]: fr for fr in json.load(f)["frases"]}
    frase = frases[int(sys.argv[1])]
    print(frase["texto"])
    print("\n--- Outlook (Graph) ---")
    crear_en_outlook(frase["ground_truth"])
    print("\n--- Google Calendar ---")
    crear_en_google(frase["ground_truth"])
    ruta = f"evento_{frase['id']}.ics"
    with open(ruta, "w", encoding="utf-8", newline="") as f:
        f.write(a_ics(frase["ground_truth"]))
    print(f"Escrito: {ruta}")

Overwriting calendario.py


In [ ]:
%%writefile resolvedor_temporal.py
# -*- coding: utf-8 -*-
"""
resolvedor_temporal.py -- Herramienta determinista de resolucion temporal.

Vinculo con el diagnostico de Deliverable 1: el piso de error del modelo
(Qwen3-4B-Instruct-2507) esta concentrado en la resolucion de fechas
relativas; por eso el calculo temporal NO lo hace el modelo de lenguaje,
sino esta herramienta, con reglas auditables y reproducibles.

La regla de oro es: si la expresion no entra en la gramatica, se devuelve
None. Nunca se inventa un valor.

Referencia temporal del proyecto: 2026-09-01 (martes).
"""

from __future__ import annotations

import re
import unicodedata
from datetime import date, datetime, timedelta

FECHA_REF = "2026-09-01"
DIA_REF = "martes"

_MESES = {
    "enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6,
    "julio": 7, "agosto": 8, "septiembre": 9, "setiembre": 9, "octubre": 10,
    "noviembre": 11, "diciembre": 12,
    "january": 1, "february": 2, "march": 3, "april": 4, "june": 6,
    "july": 7, "august": 8, "september": 9, "october": 10, "november": 11,
    "december": 12,
}
_MESES_EN = {
    "january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6,
    "july": 7, "august": 8, "september": 9, "october": 10, "november": 11,
    "december": 12,
}

# nombre normalizado -> isoweekday (1=lunes .. 7=domingo)
_DIAS = {
    "lunes": 1, "martes": 2, "miercoles": 3, "jueves": 4, "viernes": 5,
    "sabado": 6, "domingo": 7,
    "monday": 1, "tuesday": 2, "wednesday": 3, "thursday": 4, "friday": 5,
    "saturday": 6, "sunday": 7,
}

_NUMEROS = {
    "una": 1, "un": 1, "dos": 2, "tres": 3, "cuatro": 4, "cinco": 5,
    "seis": 6, "siete": 7, "ocho": 8, "nueve": 9, "diez": 10,
    "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6,
    "seven": 7, "eight": 8, "nine": 9, "ten": 10,
}

_MESES_ALT = "|".join(sorted(_MESES, key=len, reverse=True))
_MESES_EN_ALT = "|".join(sorted(_MESES_EN, key=len, reverse=True))
_DIAS_ALT = "|".join(sorted(_DIAS, key=len, reverse=True))
_NUMS_ALT = "|".join(sorted(_NUMEROS, key=len, reverse=True))

_RE_DDM = re.compile(r"\b(\d{1,2})[/-](\d{1,2})\b")
_RE_DIA_DE_MES = re.compile(r"\b(\d{1,2})\s+de\s+(" + _MESES_ALT + r")\b")
_RE_MES_DIA_EN = re.compile(r"\b(" + _MESES_EN_ALT + r")\s+(\d{1,2})(?:st|nd|rd|th)?\b")
_RE_DIA_DE_MES_EN = re.compile(r"\b(\d{1,2})(?:st|nd|rd|th)?\s+of\s+(" + _MESES_EN_ALT + r")\b")
_RE_PASADO_MANANA = re.compile(r"\bpasado\s+manana\b")
_RE_MANANA = re.compile(r"\bmanana\b")
_RE_EN_N = re.compile(r"\b(?:en|in)\s+(" + _NUMS_ALT + r"|\d+)\s+(dia|dias|day|days|semana|semanas|week|weeks)\b")
_RE_SEMANA_PROX = re.compile(r"\b(" + _DIAS_ALT + r")\s+(?:de\s+la\s+|of\s+the\s+)(?:proxima|next)\s+semana\b")
_RE_WEEKDAY = re.compile(r"\b(?:el\s+|la\s+|on\s+|this\s+|next\s+|este\s+|esta\s+|proximo\s+|proxima\s+)*(" + _DIAS_ALT + r")\b")
_RE_DIA_SOLO = re.compile(r"\bel\s+(?:dia\s+)?(\d{1,2})\b")

# Contextos que anulan una expresion (no son fechas del evento).
_EXCL_MANANA = ("de la ", "por la ", "en la ", "esta ", "la ", "the ", "this ")
_EXCL_WEEKDAY = ("los ", "las ", "cada ", "every ", "each ")

_RE_HORA_A_LAS = re.compile(
    r"\ba\s+las\s+(\d{1,2})(?::(\d{2}))?\s*"
    r"(am|pm|de\s+la\s+manana|de\s+la\s+tarde|de\s+la\s+noche)?"
)
_RE_HORA_AT = re.compile(r"\bat\s+(\d{1,2})(?::(\d{2}))?\s*(am|pm)?")
_RE_HORA_24H = re.compile(r"\b(\d{1,2}):(\d{2})\s*(am|pm)?")
_RE_HORA_MER = re.compile(r"\b(\d{1,2})\s*(am|pm)\b")
_RE_HORA_FRANJA = re.compile(r"\b(\d{1,2})\s*de\s+la\s+(manana|tarde|noche)\b")
_RE_RANGO = re.compile(r"\bde\s+(\d{1,2})\s+a\s+(\d{1,2})\b")


# ---------------------------------------------------------------------------
# utilidades
# ---------------------------------------------------------------------------

def _norm(texto: str) -> str:
    """Minusculas + sin tildes. Conserva la longitud (los spans siguen valiendo)."""
    s = texto.lower()
    s = "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")
    return s


def _iso(d: date) -> str:
    return d.isoformat()


def _a_date(iso: str) -> date:
    return datetime.strptime(iso, "%Y-%m-%d").date()


def _con_anio(dia: int, mes: int, ref: date) -> str | None:
    """Fecha valida en el anio de ref; si ya paso, anio siguiente."""
    for anio in (ref.year, ref.year + 1):
        try:
            return _iso(date(anio, mes, dia))
        except ValueError:
            continue
    return None


def _proximo_weekday(ref: date, objetivo: int) -> date:
    """Siguiente ocurrencia estrictamente posterior a ref."""
    delta = (objetivo - ref.isoweekday()) % 7
    if delta == 0:
        delta = 7
    return ref + timedelta(days=delta)


def _weekday_semana_proxima(ref: date, objetivo: int) -> date:
    lunes_prox = ref - timedelta(days=ref.isoweekday() - 1) + timedelta(days=7)
    return lunes_prox + timedelta(days=objetivo - 1)


def _previo(t: str, inicio: int, contextos: tuple) -> bool:
    antes = t[max(0, inicio - 10):inicio]
    return any(antes.endswith(c) for c in contextos)


def _a_entero(valor: str) -> int | None:
    if valor.isdigit():
        return int(valor)
    return _NUMEROS.get(valor)


# ---------------------------------------------------------------------------
# busqueda de la expresion temporal de FECHA
# ---------------------------------------------------------------------------

def _buscar_fecha(t: str, ref: date) -> tuple[str | None, tuple[int, int] | None]:
    """Devuelve (fecha_iso, span) sobre el texto normalizado `t`."""

    # 1. formato dd/mm o dd-mm  (convencion del benchmark: dia/mes)
    for m in _RE_DDM.finditer(t):
        dia, mes = int(m.group(1)), int(m.group(2))
        if 1 <= dia <= 31 and 1 <= mes <= 12:
            f = _con_anio(dia, mes, ref)
            if f:
                return f, m.span()

    # 2. "15 de septiembre" / "september 8th" / "8th of september"
    for m in _RE_DIA_DE_MES.finditer(t):
        f = _con_anio(int(m.group(1)), _MESES[m.group(2)], ref)
        if f:
            return f, m.span()
    for m in _RE_MES_DIA_EN.finditer(t):
        f = _con_anio(int(m.group(2)), _MESES_EN[m.group(1)], ref)
        if f:
            return f, m.span()
    for m in _RE_DIA_DE_MES_EN.finditer(t):
        f = _con_anio(int(m.group(1)), _MESES_EN[m.group(2)], ref)
        if f:
            return f, m.span()

    # 3. "pasado manana"  (antes que "manana", si no daria +1)
    m = _RE_PASADO_MANANA.search(t)
    if m:
        return _iso(ref + timedelta(days=2)), m.span()

    # 4. "manana" (no "de la manana", "esta manana", ...)
    for m in _RE_MANANA.finditer(t):
        if _previo(t, m.start(), _EXCL_MANANA):
            continue
        return _iso(ref + timedelta(days=1)), m.span()

    # 5. "en dos semanas" / "in two weeks"
    m = _RE_EN_N.search(t)
    if m:
        n = _a_entero(m.group(1))
        if n:
            semanas = m.group(2).startswith(("semana", "week"))
            return _iso(ref + timedelta(days=7 * n if semanas else n)), m.span()

    # 6. "miercoles de la proxima semana"
    m = _RE_SEMANA_PROX.search(t)
    if m:
        return _iso(_weekday_semana_proxima(ref, _DIAS[m.group(1)])), m.span()

    # 7. "el lunes" / "proximo martes" / "on monday"
    for m in _RE_WEEKDAY.finditer(t):
        if _previo(t, m.start(), _EXCL_WEEKDAY):
            continue
        return _iso(_proximo_weekday(ref, _DIAS[m.group(1)])), m.span()

    # 8. "el 10" (solo dia; mes corriente si queda futuro)
    for m in _RE_DIA_SOLO.finditer(t):
        dia = int(m.group(1))
        if not 1 <= dia <= 31:
            continue
        mes, anio = ref.month, ref.year
        if dia < ref.day:
            mes += 1
            if mes == 13:
                mes, anio = 1, anio + 1
        try:
            return _iso(date(anio, mes, dia)), m.span()
        except ValueError:
            # dia invalido para el mes (p.ej. 31 de febrero) -> mes siguiente
            try:
                mes2 = mes + 1 if mes < 12 else 1
                anio2 = anio if mes < 12 else anio + 1
                return _iso(date(anio2, mes2, dia)), m.span()
            except ValueError:
                continue

    return None, None


# ---------------------------------------------------------------------------
# busqueda de la expresion temporal de HORA
# ---------------------------------------------------------------------------

def _aplicar_hora(h: int, m: int, sufijo: str | None) -> str | None:
    suf = (sufijo or "").replace("  ", " ").strip()
    suf = re.sub(r"\s+", " ", suf)
    if suf in ("pm", "de la tarde", "de la noche"):
        if h < 12:
            h += 12
    elif suf in ("am", "de la manana"):
        if h == 12:
            h = 0
    else:
        # convencion del benchmark: hora en blanco 1..7 = tarde
        if 1 <= h <= 7:
            h += 12
    if h > 23 or m > 59:
        return None
    return "%02d:%02d" % (h, m)


def _buscar_hora(t: str, spans: list[tuple[int, int]]) -> tuple[str | None, tuple[int, int] | None]:
    """Primera expresion horaria del texto, ignorando los spans de fecha."""
    candidatos = []

    for m in _RE_HORA_A_LAS.finditer(t):
        candidatos.append((m, m.group(1), m.group(2) or "0", m.group(3)))
    for m in _RE_HORA_AT.finditer(t):
        candidatos.append((m, m.group(1), m.group(2) or "0", m.group(3)))
    for m in _RE_HORA_24H.finditer(t):
        candidatos.append((m, m.group(1), m.group(2), m.group(3)))
    for m in _RE_HORA_MER.finditer(t):
        candidatos.append((m, m.group(1), "0", m.group(2)))
    for m in _RE_HORA_FRANJA.finditer(t):
        suf = "de la " + m.group(2)
        candidatos.append((m, m.group(1), "0", suf))
    for m in _RE_RANGO.finditer(t):
        candidatos.append((m, m.group(1), "0", None))

    def sin_solape(span):
        a, b = span
        return not any(a < fb and b > fa for fa, fb in spans)

    candidatos = [c for c in candidatos if sin_solape(c[0].span())]
    if not candidatos:
        return None, None

    m, hs, ms, sufijo = min(candidatos, key=lambda c: (c[0].start(), -c[0].end()))
    try:
        h, mm = int(hs), int(ms)
    except ValueError:
        return None, None
    hora = _aplicar_hora(h, mm, sufijo)
    return hora, m.span()


# ---------------------------------------------------------------------------
# API publica
# ---------------------------------------------------------------------------

def analizar_temporal(texto: str, fecha_ref: str = FECHA_REF) -> dict:
    """
    Resuelve la parte temporal de un mensaje.

    Devuelve:
        fecha        -> 'YYYY-MM-DD' o None
        hora_inicio  -> 'HH:MM' o None
        fecha_expr   -> expresion literal encontrada (para trazabilidad)
        hora_expr    -> expresion literal encontrada (para trazabilidad)
    """
    ref = _a_date(fecha_ref)
    t = _norm(texto)

    fecha, span_fecha = _buscar_fecha(t, ref)
    spans = [span_fecha] if span_fecha else []
    hora, span_hora = _buscar_hora(t, spans)

    def superficie(span):
        if not span:
            return None
        return texto[span[0]:span[1]].strip()

    return {
        "fecha": fecha,
        "hora_inicio": hora,
        "fecha_expr": superficie(span_fecha),
        "hora_expr": superficie(span_hora),
    }


def resolver_fecha(texto: str, fecha_ref: str = FECHA_REF) -> str | None:
    """Solo la fecha. None si la expresion no entra en la gramatica."""
    return analizar_temporal(texto, fecha_ref)["fecha"]


def resolver_hora_expr(expr: str) -> str | None:
    """Resuelve una expresion horaria aislada ('a las 4', '10am', '15:30')."""
    if not expr:
        return None
    hora, _ = _buscar_hora(_norm(expr), [])
    if hora:
        return hora
    return resolver_hora_valor(expr)


def resolver_hora_valor(valor) -> str | None:
    """Valor suelto que viene del modelo: '4', '09:30', '10 pm'."""
    if valor is None:
        return None
    if not isinstance(valor, str):
        return None
    m = re.fullmatch(r"\s*(\d{1,2})(?::(\d{2}))?\s*(am|pm)?\s*", valor, re.I)
    if not m:
        return None
    return _aplicar_hora(int(m.group(1)), int(m.group(2) or 0), m.group(3))


def resolver_hora(texto: str, fecha_ref: str = FECHA_REF) -> str | None:
    """Solo la hora, ignorando los numeros que forman parte de la fecha."""
    return analizar_temporal(texto, fecha_ref)["hora_inicio"]


if __name__ == "__main__":
    pruebas = [
        "Reunion el lunes a las 3pm",
        "agenda reunion manana 10am porfa",
        "Oye, agendame una junta con la Dani pasado manana a las 4 en la sala B-11",
        "Reunion el miércoles de la próxima semana a las 11",
        "Cita en dos semanas a las 12:00",
        "Llamada a las 14:30",
        "Junta de emergencia",
        "Call with the client on Monday at 5pm",
        "Reunión el 10/09 a las 10 pm",
        "Reunión el lunes 7 de septiembre sin hora definida",
    ]
    for p in pruebas:
        print("%-70s -> %s" % (p, analizar_temporal(p)))

Overwriting resolvedor_temporal.py


In [ ]:
# @title 4. Subir credenciales de Google
# credentials.json: cliente OAuth "Desktop app" de Google Cloud (obligatorio para DESTINO = "google").
# token_google.json: opcional; si lo subes no se pide login (vence a los 7 dias).
import os
from google.colab import files

faltan = [f for f in ["credentials.json", "token_google.json"] if not os.path.exists(f)]
if faltan:
    print("Sube:", ", ".join(faltan))
    for nombre in files.upload():
        print("Subido:", nombre)
print("credentials.json:", "OK" if os.path.exists("credentials.json") else "FALTA")
print("token_google.json:", "OK" if os.path.exists("token_google.json") else "no (se pedira login)")

credentials.json: OK
token_google.json: OK


In [ ]:
# @title 5. Ubicar el adapter LoRA (checkpoint-300)
ORIGEN_ADAPTER = "subir archivos"  # @param ["drive", "subir archivos"]
RUTA_DRIVE = "/content/drive/MyDrive/qwen3-4b-semantic-lora/checkpoint-300"  # @param {type:"string"}
import os
from google.colab import files

NECESARIOS = ["adapter_config.json", "adapter_model.safetensors"]

if ORIGEN_ADAPTER == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    ADAPTER = RUTA_DRIVE
else:
    ADAPTER = "/content/adapter"
    os.makedirs(ADAPTER, exist_ok=True)
    faltan = [f for f in NECESARIOS if not os.path.isfile(os.path.join(ADAPTER, f))]
    if faltan:
        print("Sube:", ", ".join(faltan), "(de checkpoint-300; el .safetensors pesa ~130 MB)")
        for nombre, datos in files.upload().items():
            # Colab renombra repetidos como "adapter_config (1).json"
            destino = next((n for n in NECESARIOS if nombre.startswith(n.split(".")[0])), nombre)
            with open(os.path.join(ADAPTER, destino), "wb") as f:
                f.write(datos)
            os.remove(nombre)

for f in NECESARIOS:
    assert os.path.isfile(os.path.join(ADAPTER, f)), (
        f"Falta {f} en {ADAPTER}. Con 'drive': crea el acceso directo a qwen3-4b-semantic-lora en Mi unidad.")
print("Adapter:", ADAPTER)

Sube: adapter_config.json, adapter_model.safetensors (de checkpoint-300; el .safetensors pesa ~130 MB)


Saving adapter_config.json to adapter_config.json
Saving adapter_model.safetensors to adapter_model.safetensors
Adapter: /content/adapter


In [ ]:
# @title 6. Cargar Qwen3-4B + adapter LoRA (~8 GB, tarda unos minutos la primera vez)
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODELO_ID = "Qwen/Qwen3-4B-Instruct-2507"

t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODELO_ID)
base = AutoModelForCausalLM.from_pretrained(MODELO_ID, dtype=torch.float16, device_map="auto")
# sin merge: asi el mismo modelo sirve de baseline con model.disable_adapter()
model = PeftModel.from_pretrained(base, ADAPTER).eval()
print(f"{MODELO_ID} + LoRA ({ADAPTER}) cargado en {model.device} en {time.perf_counter() - t0:.0f} s")
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM total {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB"
      f" | usada tras cargar {torch.cuda.memory_allocated() / 2**30:.2f} GB")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Qwen/Qwen3-4B-Instruct-2507 + LoRA (/content/adapter) cargado en cuda:0 en 186 s
GPU: Tesla T4 | VRAM total 14.6 GB | usada tras cargar 7.62 GB


In [ ]:
# @title 7. Extractor fine-tuneado + resolvedor de fechas (la intervención)
import contextlib
import json
import re
from datetime import date, datetime
from zoneinfo import ZoneInfo
import resolvedor_temporal as rt

# Identico a PROMPT_SEMANTICO de pipeline/prompts.py del repo: es el prompt con que se entreno el LoRA.
PROMPT_SEMANTICO = """Eres un extractor de datos de calendario.

REGLAS:
1. Devuelve SOLO JSON con estos seis campos: titulo, hora_inicio, duracion_minutos, participantes, ubicacion, recordatorio_minutos.
2. NO calcules ni devuelvas fechas: el campo fecha lo resuelve otro componente del sistema.
3. NO devuelvas zona_horaria ni recurrence: los calcula el sistema con reglas.
4. Si un campo NO se menciona, usa null (o [] para participantes). NUNCA inventes.
5. titulo: nombre breve del evento; elimina hora, participantes y ubicacion, pero conserva sus descriptores (p. ej., "Cita médica", "Reunión de planificación"). Conserva el idioma de la frase.
6. Si la frase es un verbo de encuentro sin nombre de evento (p. ej. "quedamos el viernes", "presentamos el lunes"), usa la etiqueta nominal del evento: "quedamos" -> "Quedada", "presentamos" -> "Presentación", "comemos" -> "Almuerzo".
7. hora_inicio: HH:MM 24h. Si no hay = null.
8. duracion_minutos: entero (1 hora=60). Si no hay = null.
9. participantes: nombres tal como se mencionan, PERO quitando el articulo inicial singular ("el"/"la"/"the"): "con el cliente" -> ["cliente"], "con la Dani" -> ["Dani"], "with the team" -> ["team"]. El articulo plural o total SI se conserva: "con los clientes" -> ["los clientes"], "con todo el equipo" -> ["todo el equipo"], "con los 5 directores" -> ["los 5 directores"]. Excepcion: "el profe" se conserva entero -> ["el profe"]. Si no hay = [].
10. ubicacion: donde se realiza. Si no se menciona = null.
11. recordatorio_minutos: entero. Si no se menciona = null."""

ZONA_DEFECTO = "America/Santiago"
ZONAS = {"madrid": "Europe/Madrid", "argentina": "America/Argentina/Buenos_Aires", "santiago": "America/Santiago"}
DIAS_EN = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
_DIA_A_EN = {nombre: DIAS_EN[n - 1] for nombre, n in rt._DIAS.items()}
_DIAS_RE = "|".join(sorted(_DIA_A_EN, key=len, reverse=True))
_NUM = {"dos": 2, "tres": 3, "cuatro": 4, "cinco": 5, "seis": 6, "diez": 10,
        "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "ten": 10, "other": 2}
_NUM_RE = r"\d+|" + "|".join(_NUM)


def hoy_santiago():
    return datetime.now(ZoneInfo("America/Santiago")).date()


def _generar(mensajes, con_adapter=True, max_new_tokens=256):
    """Decodificacion greedy (determinista). con_adapter=False -> Qwen3-4B base, sin fine-tuning."""
    try:
        prompt = tokenizer.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True,
                                               enable_thinking=False)
    except TypeError:
        prompt = tokenizer.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    sin_adapter = contextlib.nullcontext() if con_adapter else model.disable_adapter()
    with torch.no_grad(), sin_adapter:
        salida = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                temperature=None, top_p=None, top_k=None,
                                pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(salida[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).strip()


def _extraer_json(respuesta):
    """Primer objeto JSON valido de la respuesta, o None."""
    decoder = json.JSONDecoder()
    for m in re.finditer(r"\{", respuesta):
        try:
            obj, _ = decoder.raw_decode(respuesta[m.start():])
            if isinstance(obj, dict):
                return obj
        except json.JSONDecodeError:
            continue
    return None


# --- normalizacion de los 6 campos del modelo (como pipeline/normalizacion.py del repo) ---

def _hora(valor):
    if not isinstance(valor, str):
        return None
    hora = rt.resolver_hora_expr(valor)
    if hora:
        return hora
    return valor.strip() if re.fullmatch(r"\d{2}:\d{2}", valor.strip()) else None


def _entero(valor):
    if isinstance(valor, bool):
        return None
    if isinstance(valor, (int, float)):
        return int(valor)
    if isinstance(valor, str):
        m = re.search(r"\d+", valor)
        return int(m.group()) if m else None
    return None


def _participantes(valor):
    if not isinstance(valor, list):
        return []
    return [x.strip() for x in valor
            if isinstance(x, str) and x.strip() and not x.strip().startswith("<") and not x.strip().isdigit()]


def _ubicacion(valor, texto):
    if isinstance(valor, str) and valor.strip():
        return valor.strip()
    t = texto.lower()
    return next((e for e in ["virtual", "videollamada", "google meet", "zoom", "teams"] if e in t), None)


def _zona(texto):
    t = texto.lower()
    return next((z for clave, z in ZONAS.items() if clave in t), ZONA_DEFECTO)


# --- reglas deterministas: recurrencia y "hoy" (el resolvedor del repo no los cubre) ---

def _quitar_hasta(texto, ref):
    """'hasta el 30 de octubre' -> (fecha de termino, texto sin esa expresion)."""
    m = re.search(r"\b(hasta|until)\b", texto, re.I)
    if not m:
        return None, texto
    resto = texto[m.end():m.end() + 30]
    a = rt.analizar_temporal(resto, ref)
    if not a["fecha"] or not a["fecha_expr"]:
        return None, texto
    fin = m.end() + resto.find(a["fecha_expr"]) + len(a["fecha_expr"])
    return a["fecha"], texto[:m.start()] + texto[fin:]


def detectar_recurrencia(texto, ref):
    """Devuelve (recurrencia del contrato o None, texto sin la expresion 'hasta ...')."""
    t = rt._norm(texto)
    dias = []
    for m in re.finditer(r"\b(?:todos\s+los|todas\s+las|cada|every|each|los)\s+((?:%s)s?(?:\s*(?:,|y|e|and)\s*(?:%s)s?)*)"
                         % (_DIAS_RE, _DIAS_RE), t):
        dias += [_DIA_A_EN[d] for d in re.findall(_DIAS_RE, m.group(1))]
    dias += [d for d in re.findall(r"\b(%s)s\b" % "|".join(DIAS_EN), t)]          # "mondays"
    dias = sorted(set(dias), key=DIAS_EN.index)

    frecuencia, intervalo = None, 1
    if dias:
        frecuencia = "weekly"
    elif re.search(r"\b(todos\s+los\s+dias|cada\s+dia|a\s+diario|diari[oa]s?|diariamente|every\s+day|daily)\b", t):
        frecuencia = "daily"
    elif re.search(r"\b(semanal(mente)?|cada\s+semana|todas\s+las\s+semanas|every\s+week|weekly)\b", t):
        frecuencia = "weekly"
    elif re.search(r"\b(mensual(mente)?|cada\s+mes|todos\s+los\s+meses|every\s+month|monthly)\b", t):
        frecuencia = "monthly"
    elif re.search(r"\bquincenal(mente)?\b", t):
        frecuencia, intervalo = "weekly", 2
    m = re.search(r"\b(?:cada|every)\s+(%s)\s+(dias?|days?|semanas?|weeks?|mes(?:es)?|months?)\b" % _NUM_RE, t)
    if m:
        intervalo = int(m.group(1)) if m.group(1).isdigit() else _NUM[m.group(1)]
        unidad = m.group(2)
        frecuencia = "daily" if unidad.startswith(("dia", "day")) else \
                     "monthly" if unidad.startswith(("mes", "month")) else "weekly"
    if frecuencia is None:
        return None, texto

    hasta, texto_sin_hasta = _quitar_hasta(texto, ref)
    veces = re.search(r"\b(%s)\s+(?:veces|times|sesiones|ocurrencias)\b" % _NUM_RE, t)
    veces = (int(veces.group(1)) if veces.group(1).isdigit() else _NUM[veces.group(1)]) if veces else None
    return {"frecuencia": frecuencia, "intervalo": intervalo,
            "dias": dias or None if frecuencia == "weekly" else None,
            "hasta": hasta, "veces": None if hasta else veces}, texto_sin_hasta


def resolver_fecha(texto, ref, recurrencia):
    fecha = rt.resolver_fecha(texto, ref)
    if fecha:
        return fecha
    if re.search(r"\b(hoy|today|tonight|esta\s+(tarde|noche|manana))\b", rt._norm(texto)):
        return ref
    if recurrencia and recurrencia.get("dias"):
        # "cada lunes a las 10": la primera ocurrencia es el proximo dia de la lista
        base = datetime.strptime(ref, "%Y-%m-%d").date()
        return min(rt._proximo_weekday(base, DIAS_EN.index(d) + 1) for d in recurrencia["dias"]).isoformat()
    return None


def extraer(texto, ref=None):
    """Mensaje -> (evento de 9 campos, info). info: salida cruda del LoRA, segundos y si hubo error."""
    ref = ref or hoy_santiago().isoformat()
    t0 = time.perf_counter()
    crudo = _generar([{"role": "system", "content": PROMPT_SEMANTICO},
                      {"role": "user", "content": 'Frase: "%s"' % texto}])
    segundos = time.perf_counter() - t0
    obj = _extraer_json(crudo)
    recurrencia, texto_fecha = detectar_recurrencia(texto, ref)
    evento = {
        "titulo": (obj or {}).get("titulo") or None,
        "fecha": resolver_fecha(texto_fecha, ref, recurrencia),
        "hora_inicio": _hora((obj or {}).get("hora_inicio")),
        "duracion_minutos": _entero((obj or {}).get("duracion_minutos")),
        "participantes": _participantes((obj or {}).get("participantes")),
        "ubicacion": _ubicacion((obj or {}).get("ubicacion"), texto),
        "recordatorio_minutos": _entero((obj or {}).get("recordatorio_minutos")),
        "zona_horaria": _zona(texto),
        "recurrencia": recurrencia,
    }
    return evento, {"crudo": crudo, "segundos": segundos, "error": obj is None}


def baseline(texto):
    """Mismo prompt directo del chat original, con Qwen3-4B base (adapter desactivado)."""
    return _generar([{"role": "user", "content": f'Agenda esta reunion en el calendario: "{texto}"'}],
                    con_adapter=False)


print("Extractor listo")

Extractor listo


In [ ]:
# @title 8. Preparar el chat
import html
from IPython.display import HTML, display
from google.colab import files
import calendario

DIAS_ES = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
DIAS_PLURAL = {"monday": "lunes", "tuesday": "martes", "wednesday": "miércoles", "thursday": "jueves",
               "friday": "viernes", "saturday": "sábados", "sunday": "domingos"}
MESES_ES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
            "septiembre", "octubre", "noviembre", "diciembre"]
UNIDADES = {"daily": ("día", "días"), "weekly": ("semana", "semanas"), "monthly": ("mes", "meses")}

CHAT = []          # historial: (quien, contenido_html) con quien en {"tu", "bot", "baseline"}
PENDIENTE = None   # evento al que le falta fecha u hora; el siguiente mensaje lo completa


def _completar(pendiente, nuevo):
    """Rellena solo los campos que le faltaban al evento pendiente."""
    r = dict(pendiente)
    for campo, valor in nuevo.items():
        if r.get(campo) in (None, []) and valor not in (None, []):
            r[campo] = valor
    return r


def _fecha_es(iso):
    f = datetime.strptime(iso, "%Y-%m-%d")
    return f"{DIAS_ES[f.weekday()]} {f.day} de {MESES_ES[f.month - 1]} de {f.year}"


def _resumen(ev):
    texto = f"<b>{html.escape(ev.get('titulo') or 'Reunión')}</b> el {_fecha_es(ev['fecha'])} a las {ev['hora_inicio']}"
    dur = ev.get("duracion_minutos")
    texto += f" ({dur} min)" if dur else f" ({calendario.DURACION_DEFECTO} min, duración por defecto)"
    if ev.get("ubicacion"):
        texto += f" en {html.escape(str(ev['ubicacion']))}"
    if ev.get("participantes"):
        texto += " con " + ", ".join(html.escape(str(p)) for p in ev["participantes"])
    texto += "."
    rec = ev.get("recurrencia")
    if rec:
        uno, varios = UNIDADES.get(rec.get("frecuencia"), UNIDADES["weekly"])
        n = int(rec.get("intervalo") or 1)
        texto += f" Se repite cada {uno}" if n == 1 else f" Se repite cada {n} {varios}"
        if rec.get("frecuencia", "weekly") == "weekly":
            dia_inicio = calendario.DIAS_SEMANA[datetime.strptime(ev["fecha"], "%Y-%m-%d").weekday()]
            dias = rec.get("dias") or [dia_inicio]
            texto += " los " + ", ".join(DIAS_PLURAL.get(d.lower(), d) for d in dias)
        if rec.get("hasta"):
            texto += f" hasta el {_fecha_es(rec['hasta'])}"
        elif rec.get("veces"):
            texto += f", {int(rec['veces'])} veces"
        texto += "."
    if ev.get("recordatorio_minutos") is not None:
        texto += f" Te aviso {int(ev['recordatorio_minutos'])} min antes."
    return texto


def _detalle_html(ev, info):
    """JSON final + lo que devolvio el LoRA (sin fecha) + tiempo de inferencia."""
    return ("<details style='margin-top:6px'><summary>JSON extraído</summary><pre style='white-space:pre-wrap'>"
            + html.escape(json.dumps(ev, indent=2, ensure_ascii=False)) + "</pre>"
            + "<div style='font-size:12px;color:#555'>Salida del LoRA (6 campos; la fecha la calcula el resolvedor, "
            + f"no el modelo) en {info['segundos']:.1f} s:</div><pre style='white-space:pre-wrap'>"
            + html.escape(info["crudo"]) + "</pre></details>")


def _mostrar():
    estilos = {
        "tu": ("flex-end", "#dcf8c6", "Tú"),
        "bot": ("flex-start", "#f1f0f0", "CalendarBot (Qwen3-4B + LoRA)"),
        "baseline": ("flex-start", "#fff4e5", "Sin CalendarBot (prompt directo, Qwen3-4B sin fine-tuning)"),
    }
    burbujas = []
    for quien, contenido in CHAT:
        lado, fondo, nombre = estilos[quien]
        borde = "1px dashed #d08a00" if quien == "baseline" else "none"
        burbujas.append(
            f"<div style='display:flex;justify-content:{lado};margin:6px 0'>"
            f"<div style='max-width:75%;background:{fondo};color:#111;border:{borde};border-radius:12px;"
            f"padding:8px 12px;font-family:sans-serif;font-size:14px'>"
            f"<div style='font-size:11px;color:#555;margin-bottom:2px'>{nombre}</div>{contenido}</div></div>")
    display(HTML("<div style='max-width:760px'>" + "".join(burbujas) + "</div>"))


print("Chat listo. Escribe tu solicitud en la celda siguiente y ejecútala.")

Chat listo. Escribe tu solicitud en la celda siguiente y ejecútala.


In [ ]:
# @title 9. Chat: escribe tu solicitud y ejecuta la celda
SOLICITUD = "Agenda la entrega de GENIA para mañana a las 23:59 desde mi casa en conjunto con Matías, Pablo y Daniela"  # @param {type:"string"}
DESTINO = "google"  # @param ["google", "solo .ics"]
MOSTRAR_BASELINE = True  # @param {type:"boolean"}
REINICIAR_CHAT = True  # @param {type:"boolean"}

if REINICIAR_CHAT:
    CHAT.clear()
    PENDIENTE = None

CHAT.append(("tu", html.escape(SOLICITUD)))
if MOSTRAR_BASELINE:
    CHAT.append(("baseline", html.escape(baseline(SOLICITUD)[:600])))

evento, info = extraer(SOLICITUD)
if PENDIENTE is not None:
    evento = _completar(PENDIENTE, evento)

if info["error"] and PENDIENTE is None:
    CHAT.append(("bot", "No entendí la solicitud. ¿Me la escribes de otra forma?" + _detalle_html(evento, info)))
elif not evento.get("fecha") or not evento.get("hora_inicio"):
    PENDIENTE = evento
    falta = " y ".join(t for t, campo in [("qué día", "fecha"), ("a qué hora", "hora_inicio")] if not evento.get(campo))
    titulo = html.escape(evento.get("titulo") or "la reunión")
    CHAT.append(("bot", f"Anoté <b>{titulo}</b>, pero me falta saber {falta}. ¿Me lo dices?" + _detalle_html(evento, info)))
else:
    PENDIENTE = None
    if DESTINO == "google":
        res = calendario.crear_en_google(evento, "credentials.json", "token_google.json", enviar=True)
    else:
        res = {"ics": f"evento_{datetime.now().strftime('%H%M%S')}.ics"}
        with open(res["ics"], "w", encoding="utf-8", newline="") as f:
            f.write(calendario.a_ics(evento))

    link = res.get("htmlLink")
    if link:
        respuesta = f"Listo, agendé {_resumen(evento)} <a href='{link}' target='_blank'>Ver en el calendario</a>"
    else:
        respuesta = f"Preparé {_resumen(evento)} Te dejo el archivo .ics para agregarlo a tu calendario."
        files.download(res["ics"])
    CHAT.append(("bot", respuesta + _detalle_html(evento, info)))

_mostrar()

Evento creado en Google Calendar: https://www.google.com/calendar/event?eid=OGhmbGFwcnViaDNvNWtucDk2NTVrdWZhYnMgYzEyZjEyNDMwN2E5MmQ0YTY2N2VkYTMxMWRjNTgwMzc4Nzk3YmQ2ZGY2NWIwZDU0ZTBiYjk0Yzg4MzJiZGJhOEBn


In [ ]:
# @title 10. (Opcional) Hardware usado en esta sesión, para el informe
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import shutil, psutil
print(f"VRAM máxima asignada por PyTorch: {torch.cuda.max_memory_allocated() / 2**30:.2f} GB")
print(f"RAM total: {psutil.virtual_memory().total / 2**30:.1f} GB | disco libre: {shutil.disk_usage('/').free / 2**30:.0f} GB")
print("dtype:", next(model.parameters()).dtype)

name, memory.total [MiB], memory.used [MiB]
Tesla T4, 15360 MiB, 8223 MiB
VRAM máxima asignada por PyTorch: 7.81 GB
RAM total: 12.7 GB | disco libre: 57 GB
dtype: torch.float16
